In [2]:
import os
import h5py
import numpy as np
import torch
import torch.nn.functional as F

# Update these paths before running.
DINOV3_LOCATION = "/root/autodl-tmp/dinov3"
MODEL_NAME = "dinov3_vits16plus"
MODEL_WEIGHTS = DINOV3_LOCATION + "/weights/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth"
FG_CLASSIFIER_PATH = DINOV3_LOCATION + "/weights/fg_classifier.pkl"
IMAGE_PATH = '../data/dog.jpeg'
OUTPUT_DIR = '../output'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model=MODEL_NAME,
    source="local",
    weights=MODEL_WEIGHTS,
)
model = model.to(device).eval()

print("Model loaded.")

Model loaded.


In [ ]:
def l2_normalize(x, dim=-1, eps=1e-6):
    norm = torch.norm(x, p=2, dim=dim, keepdim=True).clamp(min=eps)
    return x / norm

def extract_dense_features(dino, image_np, device='cuda'):
    '''
    image_np: (H, W, 3)
    returns:
        feat_map: (Hf, Wf, D) torch.Tensor
    '''
    image = torch.from_numpy(image_np).float().permute(2, 0, 1) / 255.0
    image = image.unsqueeze(0).to(device)  # (1, 3, H, W)

    # Core requirement: get patch-level / dense features, not cls token only.
    with torch.no_grad():
        out = dino.forward_features(image)

    